# Title

In [2]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
dict_attributes = set(dir(dict())) - set(dir(object()))

{'__class_getitem__',
 '__contains__',
 '__delitem__',
 '__getitem__',
 '__ior__',
 '__iter__',
 '__len__',
 '__or__',
 '__reversed__',
 '__ror__',
 '__setitem__',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values'}

## Remark - Initializing dicts

There are 3 ways of initializing `dict`

- `dict(**kwargs)`: standard key/values
- `dict(Mapping, **kwargs)`:  If a mapping object is given, then
    1. A list of keys `list[key]` will be generated via `list(iter(Mapping))`
    2. The values will be looked up via `Mapping.__getitem__(key)`
- `dict(Iterable, **kwargs)`: If the first item is an iterable, then:
    1. A `list[tuple[key, value]]` will be generated via `list(iter(Iterable))`


In [65]:
from collections.abc import Mapping, MutableMapping, Iterable, Union
from typing import Optional


def is_dunder(s: str) -> bool:
    return s.startswith("__") and s.endswith("__")


class Config(Iterable):
    def __init__(
        self, __dict__: Optional[Union[Mapping, Iterable]] = None, /, **kwargs
    ):
        super().__init__()

        if __dict__ is not None:
            assert not kwargs, f"kwargs not allowed if Mappping given!"

        items = kwargs if __dict__ is None else __dict__

        for key in items:
            value = items[key]
            if isinstance(value, Config):
                setattr(self, key, value)
            elif is_dunder(key):
                raise ValueError(f"Cannot set dunder key {key=}")
            # Recurse on Mapping
            else:
                if isinstance(value, Mapping):
                    setattr(self, key, Config(value))
                else:
                    setattr(self, key, value)

    def __str__(self):
        return self.__class__.__name__

    def __format__(self):
        return self.__class__.__name__

    def __repr__(self, nest_level: int = 0):
        print(nest_level)
        pad = r"_" * 4
        start_string = f"{self.__class__.__name__}("
        end_string = f")"

        lines = [start_string]

        for key, value in self.__dict__.items():
            if isinstance(value, Config):
                s = pad + f"{key} = {value.__repr__(nest_level+1)}"
            else:
                s = pad + f"{key} = {value}"
            lines.append(s)
        lines.append(end_string)
        result = ("\n" + pad * nest_level).join(lines)
        # print(result)
        return result

    def __len__(self):
        return self.__dict__.__len__()

    def __getitem__(self, key, from_iter=False):
        print(f"__getitem__ called from {id(self)} with {key=} and {from_iter=}")
        value = self.__dict__[key]

        if from_iter and isinstance(value, Config):
            return dict(value)
        return value

    def __iter__(self):
        print(f"__iter__ called, {id(self)=}")
        print(f"{self.__dict__=}")
        for key, value in self.__dict__.items():
            # if isinstance(value, Config):
            yield key, self.__getitem__(key, from_iter=True)

In [66]:
Config()

0


Config(
)

In [72]:
simple = Config(a=1, b=2)

dict(simple, c=1)

__iter__ called, id(self)=140082553811584
self.__dict__={'a': 1, 'b': 2}
__getitem__ called from 140082553811584 with key='a' and from_iter=True
__getitem__ called from 140082553811584 with key='b' and from_iter=True


{'a': 1, 'b': 2, 'c': 1}

In [68]:
z = Config(a=2, b=2, c=Config(x=1, y=2, z=Config(w=1, o=2)))

0
1
2


Config(
____a = 2
____b = 2
____c = Config(
________x = 1
________y = 2
________z = Config(
____________w = 1
____________o = 2
________)
____)
)

In [69]:
list(iter(z.c.z))

__iter__ called, id(self)=140082553811152
self.__dict__={'w': 1, 'o': 2}
__getitem__ called from 140082553811152 with key='w' and from_iter=True
__getitem__ called from 140082553811152 with key='o' and from_iter=True


[('w', 1), ('o', 2)]

In [70]:
list(iter(z))

__iter__ called, id(self)=140082553812880
0
1
self.__dict__={'a': 2, 'b': 2, 'c': Config(
____x = 1
____y = 2
____z = Config(
________w = 1
________o = 2
____)
)}
__getitem__ called from 140082553812880 with key='a' and from_iter=True
__getitem__ called from 140082553812880 with key='b' and from_iter=True
__getitem__ called from 140082553812880 with key='c' and from_iter=True
__iter__ called, id(self)=140082553814608
0
self.__dict__={'x': 1, 'y': 2, 'z': Config(
____w = 1
____o = 2
)}
__getitem__ called from 140082553814608 with key='x' and from_iter=True
__getitem__ called from 140082553814608 with key='y' and from_iter=True
__getitem__ called from 140082553814608 with key='z' and from_iter=True
__iter__ called, id(self)=140082553811152
self.__dict__={'w': 1, 'o': 2}
__getitem__ called from 140082553811152 with key='w' and from_iter=True
__getitem__ called from 140082553811152 with key='o' and from_iter=True


[('a', 2), ('b', 2), ('c', {'x': 1, 'y': 2, 'z': {'w': 1, 'o': 2}})]

In [71]:
dict(z)

__iter__ called, id(self)=140082553812880
0
1
self.__dict__={'a': 2, 'b': 2, 'c': Config(
____x = 1
____y = 2
____z = Config(
________w = 1
________o = 2
____)
)}
__getitem__ called from 140082553812880 with key='a' and from_iter=True
__getitem__ called from 140082553812880 with key='b' and from_iter=True
__getitem__ called from 140082553812880 with key='c' and from_iter=True
__iter__ called, id(self)=140082553814608
0
self.__dict__={'x': 1, 'y': 2, 'z': Config(
____w = 1
____o = 2
)}
__getitem__ called from 140082553814608 with key='x' and from_iter=True
__getitem__ called from 140082553814608 with key='y' and from_iter=True
__getitem__ called from 140082553814608 with key='z' and from_iter=True
__iter__ called, id(self)=140082553811152
self.__dict__={'w': 1, 'o': 2}
__getitem__ called from 140082553811152 with key='w' and from_iter=True
__getitem__ called from 140082553811152 with key='o' and from_iter=True


{'a': 2, 'b': 2, 'c': {'x': 1, 'y': 2, 'z': {'w': 1, 'o': 2}}}

In [157]:
z.__dict__

0
1


{'a': 2,
 'b': 2,
 'c': Config(
 ____x = 1
 ____y = 2
 ____z = Config(
 ________w = 1
 ____)
 )}

In [158]:
list(iter(z))

['a', 'b', 'c']

In [159]:
dict(z)

0
1


{'a': 2,
 'b': 2,
 'c': Config(
 ____x = 1
 ____y = 2
 ____z = Config(
 ________w = 1
 ____)
 )}

In [73]:
d = dict([(1, 2), (3, 4), (5, 6, 7)])

ValueError: dictionary update sequence element #2 has length 3; 2 is required

In [161]:
dict(iter(d))

TypeError: cannot convert dictionary update sequence element #0 to a sequence

In [162]:
dict(z)

0
1


{'a': 2,
 'b': 2,
 'c': Config(
 ____x = 1
 ____y = 2
 ____z = Config(
 ________w = 1
 ____)
 )}

In [90]:
z.__dict__

1
2


{'a': 2,
 'b': 2,
 'c': ____Config(____
 ____a = 1____
 ____b = 2____
 ____c = ________Config(________
 ____a = 1________
 )____
 )}

In [2]:
from typing import NamedTuple

In [12]:
class MyTup(NamedTuple):
    count: int
    index: float

In [14]:
MyTup(count=2, index=3)

MyTup(count=2, index=3)

In [17]:
MyTup(count=2, index=3).count

2

In [20]:
import torch